# 🌐 Federated Learning Training — Flower (FedAvg)
**Privacy-Preserving Cancer Detection Project**

This notebook uses the **Flower (`flwr`)** framework to run
Federated Learning across 3 hospital nodes using the `FedAvg` strategy.

It uses the project's `federated_learning/` module with `flwr run`.

---
### Before running:
1. Go to **Runtime → Change runtime type → T4 GPU → Save**
2. Upload these to **MyDrive/FL_Project/** on Google Drive:
   - `ml/` folder (model.py)
   - `federated_learning/` folder (client_app.py, server_app.py, task.py, pyproject.toml)
   - `data/` folder (node1/, node2/, node3/, test/)
3. Run all cells **top to bottom** (Runtime → Run all)

---

## Step 1 — Mount Google Drive

In [1]:
from google.colab import drive
drive.mount("/content/drive")
print("✅  Drive mounted.")

Mounted at /content/drive
✅  Drive mounted.


## Step 2 — Install Flower

In [2]:
!pip install -q 'flwr[simulation]==1.32.1' hatchling

import flwr
print(f"✅  Flower version: {flwr.__version__}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 974.5/974.5 kB 57.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.7/77.7 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 124.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 109.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.5/310.5 kB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.3/253.3 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.9

## Step 3 — Check what files exist on Drive

In [3]:
import os

base = "/content/drive/MyDrive/FL_Project"
print(f"Scanning {base} ...")
print()

for root, dirs, files in os.walk(base):
    level = root.replace(base, "").count(os.sep)
    indent = "  " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = "  " * 2 * (level + 1)
    for file in files[:5]:
        print(f"{subindent}{file}")
    if len(files) > 5:
        print(f"{subindent}... and {len(files)-5} more")

Scanning /content/drive/MyDrive/FL_Project ...

FL_Project/
    models/
        centralized_model.h5
        federated_model.h5
        federated_augmented.h5
    results/
        centralized_metrics.json
        fl_metrics.json
        all_metrics.json
        comparison.png
        f1_per_class.png
        ... and 2 more
    data/
        class_map.json
        sample_grid.png
        X_all.npy
        y_all.npy
        augmentation_preview.png
        node1/
            y.npy
            X.npy
            X_aug.npy
            y_aug.npy
        node2/
            X.npy
            y.npy
            X_aug.npy
            y_aug.npy
        test/
            y.npy
            X.npy
        node3/
            X.npy
            y.npy
            X_aug (1).npy
            X_aug.npy
            y_aug (1).npy
            ... and 1 more
    federated_learning/
        __init__.py
        task.py
        model.py
        pyproject.toml
        server_app.py
        ... and 1 more
        __py

## Step 4 — Copy everything from Drive to Colab VM

In [ ]:
import shutil, os

PROJECT_VM = "/content/FL_Project"
DRIVE_PATH = "/content/drive/MyDrive/FL_Project"

os.makedirs(PROJECT_VM, exist_ok=True)

# Folders to copy
for folder in ["ml", "federated_learning", "data"]:
    src = os.path.join(DRIVE_PATH, folder)
    dst = os.path.join(PROJECT_VM, folder)
    if os.path.exists(src):
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f"  ✅  Copied {folder}/")
    else:
        print(f"  ❌  {folder}/ NOT FOUND on Drive!")

# Create output dirs
os.makedirs(f"{PROJECT_VM}/models", exist_ok=True)
os.makedirs(f"{PROJECT_VM}/results", exist_ok=True)
print()
print("✅  All files copied to Colab VM.")

  ✅  Copied ml/
  ✅  Copied federated_learning/
  ✅  Copied data/

✅  All files copied to Colab VM.


## Step 5 — Verify all required files

In [6]:
import os

required_files = [
    "ml/model.py",
    "federated_learning/task.py",
    "federated_learning/client_app.py",
    "federated_learning/server_app.py",
    "federated_learning/pyproject.toml",
    "data/node1/X.npy",
    "data/node1/y.npy",
    "data/node2/X.npy",
    "data/node2/y.npy",
    "data/node3/X.npy",
    "data/node3/y.npy",
    "data/test/X.npy",
    "data/test/y.npy",
]

all_ok = True
for f in required_files:
    path = os.path.join("/content/FL_Project", f)
    exists = os.path.exists(path)
    status = "✅" if exists else "❌ MISSING"
    if not exists:
        all_ok = False
    print(f"  [{status}] {f}")

if not all_ok:
    raise FileNotFoundError("Some files are missing! Upload them to MyDrive/FL_Project/ on Google Drive.")
print("\n✅  All files present. Ready to train!")

  [✅] ml/model.py
  [✅] federated_learning/task.py
  [✅] federated_learning/client_app.py
  [✅] federated_learning/server_app.py
  [✅] federated_learning/pyproject.toml
  [✅] data/node1/X.npy
  [✅] data/node1/y.npy
  [✅] data/node2/X.npy
  [✅] data/node2/y.npy
  [✅] data/node3/X.npy
  [✅] data/node3/y.npy
  [✅] data/test/X.npy
  [✅] data/test/y.npy

✅  All files present. Ready to train!


## Step 6 — Configure GPU Memory Growth

Prevent OOM errors by enabling GPU memory growth before training.

In [7]:
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"✅  GPU found: {gpus[0].name} — memory growth enabled")
else:
    print("⚠️  No GPU found — training will be on CPU (slower)")

✅  GPU found: /physical_device:GPU:0 — memory growth enabled


In [8]:
import os

PROJECT_VM = "/content/FL_Project"

toml_content = """[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[project]
name = "cancer-detection-fl"
version = "0.1.0"
description = "Federated Learning for Privacy-Preserving Cancer Detection"
requires-python = ">=3.10,<4.0"
dependencies = [
    "flwr[simulation]>=1.26",
    "tensorflow>=2.13",
    "numpy>=1.24",
]

[tool.hatch.build.targets.wheel]
packages = ["federated_learning", "ml"]

[tool.flwr.app]
publisher = "Rohith"

[tool.flwr.app.components]
serverapp = "federated_learning.server_app:app"
clientapp = "federated_learning.client_app:app"

[tool.flwr.app.config]
num-server-rounds = 10
local-epochs = 3
batch-size = 32
learning-rate = 0.001
fraction-train = 0.67
save-model = true

[tool.flwr.federations]
default = "local-simulation"

[tool.flwr.federations.local-simulation]
options.num-supernodes = 3
"""

with open(f"{PROJECT_VM}/pyproject.toml", "w") as f:
    f.write(toml_content)

print("✅ Generated correct pyproject.toml at the root!")


✅ Generated correct pyproject.toml at the root!


## Step 7 — Run Federated Learning Training (Flower FedAvg)

This uses `flwr run` to execute the FL pipeline defined in
`federated_learning/` (client_app.py + server_app.py + pyproject.toml).

It runs **10 rounds of FedAvg** across 3 simulated hospital nodes.

In [9]:
import os, subprocess, time, sys

os.chdir("/content/FL_Project")

# 1. Patch source files for compatibility if needed
def fix_source_code(file_path):
    with open(file_path, "r") as f:
        content = f.read()

    # Fix function signatures to accept both grid and context
    content = content.replace("def main(context: Context)", "def main(grid, context: Context)")
    content = content.replace("def main(context)", "def main(grid, context)")

    # Fix relative task import to be absolute
    content = content.replace("from task import", "from federated_learning.task import")

    with open(file_path, "w") as f:
        f.write(content)

fix_source_code("federated_learning/server_app.py")
fix_source_code("federated_learning/client_app.py")

print("=" * 60)
print("  Direct Simulation Trigger (Flower FedAvg)")
print("=" * 60)

# 2. Cleanup hung processes
subprocess.run(["pkill", "-9", "-f", "flwr"], stderr=subprocess.DEVNULL)
time.sleep(2)

# 3. Create the bridge script
run_script = """
import flwr as fl
from federated_learning.server_app import app as server_app
from federated_learning.client_app import app as client_app
from flwr.simulation.run_simulation import run_simulation

run_simulation(
    server_app=server_app,
    client_app=client_app,
    num_supernodes=3,
    backend_config={"client_resources": {"num_cpus": 2, "num_gpus": 0.3}},
)
"""

with open("run_sim.py", "w") as f:
    f.write(run_script)

# 4. Execute with streaming output
env = os.environ.copy()
env["PYTHONPATH"] = "."

print("Starting simulation (10 rounds)... This will take a few minutes.")
try:
    process = subprocess.Popen(
        ["python", "run_sim.py"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env=env
    )

    for line in process.stdout:
        print(line, end="")

    process.wait()

    if process.returncode == 0:
        print("\n" + "=" * 60)
        print("  ✅  Federated Learning Successful!")
        print("=" * 60)
        subprocess.run(["ls", "-R", "models", "results"])
    else:
        print(f"\n❌  Simulation failed with exit code {process.returncode}")

except Exception as e:
    print(f"Critical Error: {e}")

  Direct Simulation Trigger (Flower FedAvg)
Starting simulation (10 rounds)... This will take a few minutes.

            This is a deprecated feature. It will be removed
            entirely in future versions of Flower.
        
/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2026-07-22 14:53:36.663102: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1784732016.664570    2187 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute cap

## Step 8 — Verify model was created

In [10]:
import os

# Verification of generated files after simulation
model_path = "/content/FL_Project/models/federated_model.h5"
metrics_path = "/content/FL_Project/results/fl_metrics.json"

print("Checking generated files:")
for path in [model_path, metrics_path]:
    if os.path.exists(path):
        size = os.path.getsize(path) / (1024 * 1024)
        print(f"  ✅  {os.path.basename(path)} ({size:.2f} MB) found at {path}")
    else:
        # Let's search if they were saved in the root instead due to path differences
        alt_path = os.path.join("/content/FL_Project", os.path.basename(path))
        if os.path.exists(alt_path):
            size = os.path.getsize(alt_path) / (1024 * 1024)
            print(f"  ✅  {os.path.basename(path)} ({size:.2f} MB) found at {alt_path}")
        else:
            print(f"  ❌  {os.path.basename(path)} NOT FOUND")

Checking generated files:
  ✅  federated_model.h5 (84.90 MB) found at /content/FL_Project/models/federated_model.h5
  ✅  fl_metrics.json (0.00 MB) found at /content/FL_Project/results/fl_metrics.json


## Step 9 — Copy model + metrics back to Google Drive

In [15]:
import shutil, os

# Source paths (checking both standard and project root)
src_model = "/content/FL_Project/models/federated_model.h5" if os.path.exists("/content/FL_Project/models/federated_model.h5") else "/content/FL_Project/federated_model.h5"
src_metrics = "/content/FL_Project/results/fl_metrics.json" if os.path.exists("/content/FL_Project/results/fl_metrics.json") else "/content/FL_Project/fl_metrics.json"

items_to_save = {
    src_model: "/content/drive/MyDrive/FL_Project/models/federated_model.h5",
    src_metrics: "/content/drive/MyDrive/FL_Project/results/fl_metrics.json",
}

print("Syncing results to Google Drive...")
for src, dst in items_to_save.items():
    if os.path.exists(src):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(src, dst)
        size = os.path.getsize(src) / (1024 * 1024)
        print(f"  ✅  {os.path.basename(src)} ({size:.2f} MB) → Drive SUCCESS")
    else:
        print(f"  ❌  Source {src} not found, skipping.")

print("\nAll project requirements met. You can now download the model from Google Drive.")

Syncing results to Google Drive...
  ✅  federated_model.h5 (84.90 MB) → Drive SUCCESS
  ✅  fl_metrics.json (0.00 MB) → Drive SUCCESS

All project requirements met. You can now download the model from Google Drive.


## Step 10 — Display metrics

In [16]:
import json, os

# Source path for metrics
metrics_path = "/content/FL_Project/results/fl_metrics.json"

if os.path.exists(metrics_path):
    with open(metrics_path) as f:
        metrics = json.load(f)

    print("=" * 60)
    print("  Final Federated Learning Results")
    print("=" * 60)

    # Display high-level metrics
    if "history" in metrics:
        history = metrics["history"]
        if "metrics_distributed" in history:
            dist_metrics = history["metrics_distributed"]
            for metric_name, values in dist_metrics.items():
                final_val = values[-1][1]
                print(f"  Final Distributed {metric_name.capitalize()}: {final_val:.4f}")

        if "losses_distributed" in history:
            final_loss = history["losses_distributed"][-1][1]
            print(f"  Final Distributed Loss     : {final_loss:.4f}")

    # Fallback for flat dictionary
    else:
        for k, v in metrics.items():
            if isinstance(v, (int, float)):
                print(f"  {k:<26}: {v:.4f}")
            elif isinstance(v, str):
                print(f"  {k:<26}: {v}")

    print("=" * 60)
    print("✅ Results are available in /content/drive/MyDrive/FL_Project/results/")
else:
    print(f"❌ Metrics file not found at {metrics_path}")

  Final Federated Learning Results
  num_rounds                : 10.0000
  fraction_train            : 0.6700
  learning_rate             : 0.0010
✅ Results are available in /content/drive/MyDrive/FL_Project/results/
